# 03 · Python — you do Rung 1 *yourself*  ·  *optional coda*

Every resolved edge in notebook 01 came from **rustdoc** — the Rust compiler's own homework. Python has no
compiler waiting to hand you resolved types. So you do Rung 1 yourself, **statically, no runtime**, with
[Griffe](https://mkdocstrings.github.io/griffe/) — and, crucially, you stay *honest about what you can't
resolve*. That's exactly where a code graph stops being a convenience and becomes the only road there.

We point it at **pydantic** (every Python dev knows `BaseModel`). Same recipe as the Rust side:
**extract → resolve → (graph → traverse)**.\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bcallender/agent-context-workshop/blob/main/notebooks/03_python_griffe_resolution.ipynb)  ·  *zero-install: click, then run the cells top to bottom.*

In [1]:
# Colab bootstrap — no-op locally. In Colab: clone the repo, install it, hydrate the data.
import sys, os
if "google.colab" in sys.modules:
    import subprocess
    REPO = "/content/agent-context-workshop"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "https://github.com/bcallender/agent-context-workshop.git", REPO], check=True)
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    subprocess.run(["bash", "scripts/setup_data.sh"], check=True)
    print("Colab ready — cloned, installed, data hydrated.")

In [2]:
import os
from pathlib import Path
from collections import Counter

from dotenv import load_dotenv

from context_workshop.parsers.python_griffe import load_python_package, extract_python_symbols

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")
HAS_KEY = bool(os.getenv("OPENROUTER_API_KEY"))

# Rung 1, done by us — statically, no runtime: load the package and resolve its re-exports (aliases).
module = load_python_package("pydantic")           # resolve_aliases runs inside (implicit=False, external=False)
syms = extract_python_symbols(module, public_only=True)
print(f"griffe extracted {len(syms)} public symbols | kinds: {dict(Counter(s.kind for s in syms))}")

griffe extracted 2372 public symbols | kinds: {'module': 72, 'reexport': 207, 'attribute': 365, 'class': 346, 'method': 950, 'function': 432}


## The collision — again, in Python

Notebook 01 opened on two Rust `PostingList`s. Here it is in Python: **two `BaseModel`s** — the v2 one and
the bundled-v1 compat one. Same name, different modules. grep can't tell them apart; griffe resolves each to
its **distinct canonical path.**

In [3]:
bms = [s for s in syms if s.name == "BaseModel" and s.is_reexport]
print(f"name == 'BaseModel' -> {len(bms)} distinct re-exports, each resolved to its canonical definition:")
for s in bms:
    print(f"  {s.qualified_name:28} ->  {s.canonical_path}")

name == 'BaseModel' -> 2 distinct re-exports, each resolved to its canonical definition:
  pydantic.v1.BaseModel        ->  pydantic.v1.main.BaseModel
  pydantic.BaseModel           ->  pydantic.main.BaseModel


## Follow the re-export — and be honest where you can't

griffe follows `from .main import BaseModel` to the real definition (`pydantic.main.BaseModel`). It does this
for *most* of the public surface — but Python is dynamic, and some aliases **can't be resolved statically**.
A good Rung-1 layer reports those gaps instead of guessing.

In [4]:
reexports = [s for s in syms if s.is_reexport]
resolved = [s for s in reexports if s.canonical_path]
unresolved = [s for s in reexports if not s.canonical_path]
print(f"re-exports: {len(reexports)}  |  resolved: {len(resolved)}  |  UNRESOLVED (honest gaps): {len(unresolved)}\n")
print("resolved (public name -> canonical definition):")
for s in resolved[:4]:
    print(f"  {s.qualified_name:32} ->  {s.canonical_path}")
print("\nunresolved — static analysis can't follow these (dynamic / conditional / external):")
for s in unresolved[:6]:
    print(f"  {s.qualified_name}  ->  None")

re-exports: 207  |  resolved: 200  |  UNRESOLVED (honest gaps): 7

resolved (public name -> canonical definition):
  pydantic.VERSION                 ->  pydantic.version.VERSION
  pydantic.AliasChoices            ->  pydantic.aliases.AliasChoices
  pydantic.AliasGenerator          ->  pydantic.aliases.AliasGenerator
  pydantic.AliasPath               ->  pydantic.aliases.AliasPath

unresolved — static analysis can't follow these (dynamic / conditional / external):
  pydantic.FieldSerializationInfo  ->  None
  pydantic.SerializationInfo  ->  None
  pydantic.SerializerFunctionWrapHandler  ->  None
  pydantic.ValidationInfo  ->  None
  pydantic.ValidatorFunctionWrapHandler  ->  None
  pydantic.v1.typing.typing_base  ->  None


## A blast-radius taste — *what depends on this?*

The question grep structurally can't answer. With the re-exports resolved, even plain **inheritance** gives a
real blast radius: change a base class, and every subclass is affected — in one pass. *(We match on the base
name here; the full graph would canonicalize these too, exactly like notebook 01.)*

In [5]:
classes = [s for s in syms if s.kind == "class"]
subclasses = {}
for c in classes:
    for b in (c.bases or []):
        subclasses.setdefault(b.split(".")[-1], []).append(c.qualified_name)
target, kids = max(subclasses.items(), key=lambda kv: len(kv[1]))
print(f"blast radius of `{target}` (direct subclasses): {len(kids)} types would be touched if it changes")
for q in sorted(kids)[:8]:
    print(f"  {q}")
print("  ...")

blast radius of `PydanticValueError` (direct subclasses): 48 types would be touched if it changes
  pydantic.v1.errors.AnyStrMaxLengthError
  pydantic.v1.errors.AnyStrMinLengthError
  pydantic.v1.errors.ColorError
  pydantic.v1.errors.DateError
  pydantic.v1.errors.DateNotInTheFutureError
  pydantic.v1.errors.DateNotInThePastError
  pydantic.v1.errors.DateTimeError
  pydantic.v1.errors.DecimalIsNotFiniteError
  ...


## Rung 2 — meaning, via fenic *(optional)*

Same as the Rust coda: fold the docstring's *intent* onto the resolved symbols with a small, focused
`semantic.map`. Needs `OPENROUTER_API_KEY`; renders an example without one.

In [6]:
from context_workshop.parsers.schema import to_arrow

docd = [s for s in syms if s.kind in ("class", "function") and s.docstring][:6]
EXAMPLE = [
    {"qualified_name": "pydantic.main.BaseModel", "summary": "Base class for models with validated, typed fields and (de)serialization."},
    {"qualified_name": "pydantic.fields.Field",   "summary": "Declares field metadata, defaults, and validation constraints on a model field."},
]
if HAS_KEY:
    import fenic as fc
    sem = fc.SemanticConfig(
        language_models={"mini": fc.OpenRouterLanguageModel(model_name="openai/gpt-4o-mini", rpm=500, tpm=200_000)},
        default_language_model="mini")
    sess = fc.Session.get_or_create(fc.SessionConfig(app_name="py_rung2", semantic=sem,
                                                     db_path=str(ROOT/"data/cache/fenic")))
    df = (sess.create_dataframe(to_arrow(docd))
          .with_column("blurb", fc.text.jinja("Python {{k}} `{{qn}}`\n{{d}}", strict=False,
              k=fc.col("kind"), qn=fc.col("qualified_name"), d=fc.col("docstring")))
          .with_column("summary", fc.semantic.map("In one concise sentence, what is this Python item for?\n\n{{b}}", b=fc.col("blurb"))))
    rows = df.select("qualified_name", "summary").to_pylist()
else:
    rows = EXAMPLE
    print("No OPENROUTER_API_KEY — showing EXAMPLE enriched rows (set the key to run fenic live).\n")
for r in rows[:4]:
    print(f"{r['qualified_name'].split('.')[-1]:18} -> {r['summary']}")

No OPENROUTER_API_KEY — showing EXAMPLE enriched rows (set the key to run fenic live).

BaseModel          -> Base class for models with validated, typed fields and (de)serialization.
Field              -> Declares field metadata, defaults, and validation constraints on a model field.


## Where this goes

Extract → **resolve** → graph → traverse. We did the first two live; loading these resolved symbols + their
edges into Neo4j is the *same recipe* as notebook 01's Rust graph — only here the resolution is something you
**built**, because nothing was going to hand it to you. In Rust the graph looked like a convenience. In
Python it's the only road to *"what depends on this?"* — which is why it's where this earns its keep first.